In [1]:
# SELF-CONTAINED: run this cell directly after restarting the kernel.
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
import pickle, json, hashlib, time, random
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO
import numpy as np
import jax
jax.config.update('jax_enable_x64',True)
import torch
from joblib import Parallel, delayed, parallel_config
from WERM.experiment_utils import weighted_erm_joint_cv
import WERM.experiment_utils as el_utils
import WERM.estimators as el_estimators
import WERM.config as el_config

EL_ROOT=Path('results/E_L_upperbound_gap10_n1000_p100')
EL_ROOT.mkdir(parents=True,exist_ok=True)
EL_SETTINGS=[(1000,100)]
EL_TRIALS=int(os.environ.get('EL_TRIALS','100'))
EL_JOBS=int(os.environ.get('EL_JOBS','36'))
EL_SEED=20261120
EL_BOUNDS=np.arange(1,7)
EL_LOSS=lambda prediction,truth:.5*(np.asarray(prediction)-np.asarray(truth))**2
EL_SCORE=lambda prediction,truth:np.abs(np.asarray(prediction)-np.asarray(truth))

def el_data(n,p,trial):
    rng=np.random.default_rng(np.random.SeedSequence([EL_SEED,n,p,trial]))
    beta=np.empty((p,4)); beta[:,0]=rng.normal(size=p)*np.sqrt(n/p)
    for k in range(1,4):beta[:,k]=beta[:,k-1]+1.5*np.sqrt(n/p)*rng.normal(size=p)
    X=rng.normal(size=(n,p))/np.sqrt(n)
    truth=np.rint(n*np.array([.2,.5,.8])).astype(int)
    state=np.searchsorted(truth,np.arange(n),side='right')
    mu=np.einsum('ij,ij->i',X,beta[:,state].T)
    y=mu+rng.normal(0,.5,n)
    return X,y,beta,truth,state,mu

def el_metrics(cp,truth,n):
    cp=np.asarray(cp,int)
    if cp.size:
        d=np.abs(cp[:,None]-truth[None,:])/n
        haus=float(max(d.min(axis=0).max(),d.min(axis=1).max()))
    else:haus=np.nan
    return dict(haus=haus,bounded_haus=haus if cp.size else 1.,empty=not bool(cp.size),
                count_error=abs(len(cp)-len(truth)),correct_count=len(cp)==len(truth))

# Cache only deterministic reference-grid enumeration, not fitted marginal
# curves or estimator outputs. GP optimization and all RNG calls are unchanged.
def el_enable_enumeration_cache():
    if not hasattr(el_config,'_el_original_enumeration'):
        el_config._el_original_enumeration=el_config.unif_prior_to_η_ϕ
        el_config._el_enumeration_cache={}
    def cached_enumeration(Lmin,Lmax,Delta,n,p_l=None):
        key=(int(Lmin),int(Lmax),int(Delta),int(n),None if p_l is None else tuple(np.asarray(p_l)))
        if key not in el_config._el_enumeration_cache:
            el_config._el_enumeration_cache[key]=el_config._el_original_enumeration(Lmin,Lmax,Delta,n,p_l=p_l)
        return el_config._el_enumeration_cache[key]
    el_config.unif_prior_to_η_ϕ=cached_enumeration

# State-aware replay cache: identical arguments AND entry RNG states are
# required for a hit; exit states are restored exactly. Thus repeated calls
# consume the same random stream and return the same original curve fit.
def el_enable_curve_replay_cache():
    if not hasattr(el_config,'_el_original_extrapolation'):
        el_config._el_original_extrapolation=el_config.extrapolated_marginals
    el_config._el_curve_replay_cache={}
    def replay_extrapolation(Lmin,Lmax,Delta_small,n_small=50,n_large=1000,p_l=None,verbose='summary'):
        entry=(np.random.get_state(),torch.get_rng_state(),random.getstate())
        arguments=(Lmin,Lmax,Delta_small,n_small,n_large,None if p_l is None else tuple(p_l))
        key=hashlib.sha256(pickle.dumps((arguments,entry[0],entry[1].numpy().tobytes(),entry[2]))).hexdigest()
        if key in el_config._el_curve_replay_cache:
            value,exit_state=el_config._el_curve_replay_cache[key]
            np.random.set_state(exit_state[0]);torch.set_rng_state(exit_state[1]);random.setstate(exit_state[2])
            return value.copy()
        value=el_config._el_original_extrapolation(Lmin,Lmax,Delta_small,n_small,n_large,p_l=p_l,verbose=verbose)
        exit_state=(np.random.get_state(),torch.get_rng_state(),random.getstate())
        el_config._el_curve_replay_cache[key]=(value.copy(),exit_state)
        return value
    el_config.extrapolated_marginals=replay_extrapolation

def el_trial(n,p,trial):
    path=EL_ROOT/f'n{n}_p{p}_trial{trial:03d}.pkl'
    if path.exists():
        with path.open('rb') as f:return pickle.load(f)
    el_enable_enumeration_cache()
    el_enable_curve_replay_cache()
    X,y,beta,truth,state,mu=el_data(n,p,trial)
    fits={}; trace={}; refits=[]; train_cp=[]; residuals=[]
    original_fit=el_utils.fit_weighted
    original_erm=el_estimators.weighted_erm
    def traced_fit(Xs,ys,*args,**kwargs):
        result=original_fit(Xs,ys,*args,**kwargs)
        refits.append(dict(rows=len(ys),norm=float(np.linalg.norm(result.coef)),
                           training_mse=float(np.mean((Xs@result.coef-ys)**2))))
        return result
    def traced_erm(*args,**kwargs):
        result=original_erm(*args,**kwargs)
        if len(kwargs.get('y',[]))<n:
            train_cp.append(dict(K=kwargs['L'],train_n=len(kwargs['y']),cp=np.asarray(result,int)))
        return result
    def traced_scorer(prediction,truth):
        residuals.extend((np.asarray(prediction)-np.asarray(truth)).reshape(-1).tolist())
        return EL_SCORE(prediction,truth)
    for upper in EL_BOUNDS:
        upper_start=time.perf_counter()
        # Reset stochastic marginal-curve optimizer restarts, so candidate fits
        # are identical when they recur under a larger upper bound.
        np.random.seed(EL_SEED+trial);torch.manual_seed(EL_SEED+trial)
        audit=trial==0 and upper==6
        if audit:
            el_utils.fit_weighted=traced_fit
            el_estimators.weighted_erm=traced_erm
        try:
            with redirect_stdout(StringIO()),redirect_stderr(StringIO()):
                (selected,_),cp,diagnostic=weighted_erm_joint_cv(
                    X=X,y=y,M=EL_LOSS,M_scorer=traced_scorer if audit else EL_SCORE,
                    Lmax=int(upper),delta_fraction=1/10,
                    l1_constants=(0.,),penalty_norm='none',coefficient_loss='least_squares',
                    num_folds=10,n_jobs=1,standardize_y=False,
                    selection_strategy='exhaustive',return_diagnostics=True)
        finally:
            el_utils.fit_weighted=original_fit;el_estimators.weighted_erm=original_erm
        if trial==0:print(f'Diagnostic n={n} p={p} upper={upper}: {time.perf_counter()-upper_start:.1f}s',flush=True)
        assert len(cp)+1==selected and selected<=upper
        fits[int(upper)]=dict(selected=int(selected),cp=np.asarray(cp,int),
                             scores=np.asarray(diagnostic['scores']),**el_metrics(cp,truth,n))
    for upper in EL_BOUNDS[:-1]:
        np.testing.assert_allclose(fits[int(upper)]['scores'],fits[6]['scores'][:upper],rtol=1e-10,atol=1e-8)
    if trial==0:
        r=np.asarray(residuals).reshape(6,10,n//10)
        np.testing.assert_allclose(np.abs(r).sum(axis=(1,2)),fits[6]['scores'],rtol=1e-12)
        assert len(train_cp)==60 and len(refits)==210
        oracle_abs=[];oracle_rows=[];oracle_rank=[];oracle_cond=[]
        for fold in range(10):
            test=np.arange(fold,n,10);train=np.setdiff1d(np.arange(n),test)
            pred=np.empty(len(test))
            for k in range(4):
                tr=train[state[train]==k];te=np.flatnonzero(state[test]==k)
                coef,_,rank,singular=np.linalg.lstsq(X[tr],y[tr],rcond=None)
                pred[te]=X[test[te]]@coef
                oracle_rows.append(len(tr));oracle_rank.append(int(rank))
                oracle_cond.append(float(singular[0]/singular[-1]))
            oracle_abs.extend(np.abs(pred-y[test]))
        trace=dict(residuals=r,fold_absolute=np.abs(r).sum(axis=2),refits=refits,train_cp=train_cp,
                   oracle_absolute_mean=float(np.mean(oracle_abs)),oracle_train_rows=oracle_rows,
                   oracle_rank=oracle_rank,oracle_condition=oracle_cond,
                   known_beta_absolute_mean=float(np.abs(mu-y).mean()),
                   true_sizes=np.bincount(state),minimum_gap_full=n//10,
                   minimum_gap_train=int(.9*n/10))
    row=dict(n=n,p=p,trial=trial,fits=fits,trace=trace,truth=truth)
    temporary=path.with_suffix(f'.{os.getpid()}.tmp')
    with temporary.open('wb') as f:pickle.dump(row,f)
    temporary.replace(path)
    return row

def el_verify_replay_cache():
    X,y,*_=el_data(500,100,0)
    kwargs=dict(X=X,y=y,M=EL_LOSS,M_scorer=EL_SCORE,Lmax=4,delta_fraction=1/10,
        l1_constants=(0.,),penalty_norm='none',coefficient_loss='least_squares',num_folds=10,n_jobs=1,
        standardize_y=False,selection_strategy='exhaustive',return_diagnostics=True)
    out=[];states=[]
    for enabled in [False,True]:
        if not enabled:
            el_config.unif_prior_to_η_ϕ=getattr(el_config,'_el_original_enumeration',el_config.unif_prior_to_η_ϕ)
            el_config.extrapolated_marginals=getattr(el_config,'_el_original_extrapolation',el_config.extrapolated_marginals)
        else:el_enable_enumeration_cache();el_enable_curve_replay_cache()
        np.random.seed(73);torch.manual_seed(73);random.seed(73)
        with redirect_stdout(StringIO()),redirect_stderr(StringIO()):out.append(weighted_erm_joint_cv(**kwargs))
        states.append((np.random.get_state(),torch.get_rng_state(),random.getstate()))
        print('Whole CV complete; replay cache=',enabled,flush=True)
    assert out[0][0]==out[1][0]
    np.testing.assert_array_equal(out[0][1],out[1][1])
    np.testing.assert_array_equal(out[0][2]['scores'],out[1][2]['scores'])
    np.testing.assert_array_equal(states[0][0][1],states[1][0][1]);assert states[0][0][2:]==states[1][0][2:]
    assert torch.equal(states[0][1],states[1][1]);assert states[0][2]==states[1][2]
    print('Whole 10-fold CV scores and selected partition bitwise identical; all RNG exit states identical.')
    with (EL_ROOT/'replay_cache_verification.pkl').open('wb') as f:
        pickle.dump(dict(cv_scores_bitwise_equal=True,selected_cp_bitwise_equal=True,all_rng_exit_states_equal=True,
            scores=out[0][2]['scores'],cp=out[0][1]),f)


smoke=el_trial(1000,100,20)
assert smoke['trial']==20
print('Fresh-trial smoke passed for all six bounds.')

/scratch/ga442/gab-code-env/papers/WeightedERM-reference/.venv-phase1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fresh-trial smoke passed for all six bounds.
